# CSE 144 Final Project — Transfer Learning

**Team members:** [Your names here]

**Instructions:**
1. Upload your Kaggle dataset zip to Google Drive first
2. Run cells top to bottom
3. After training, download `submission.csv` and submit to Kaggle


## 0. Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile

# ── 改这里：你的数据集 zip 在 Google Drive 里的路径 ──
ZIP_PATH = '/content/drive/MyDrive/cse144_dataset.zip'
EXTRACT_DIR = '/content/dataset'

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Dataset extracted!')
else:
    print('Dataset already extracted, skipping.')

TRAIN_DIR = os.path.join(EXTRACT_DIR, 'train')
TEST_DIR  = os.path.join(EXTRACT_DIR, 'test')
SAMPLE_SUBMISSION = os.path.join(EXTRACT_DIR, 'sample_submission.csv')

print('train:', os.listdir(TRAIN_DIR)[:5], '...')
print('test files:', len(os.listdir(TEST_DIR)))

## 1. Imports & Reproducibility Seed

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms, models
from PIL import Image
import pandas as pd
from pathlib import Path

# ── 固定随机种子，保证可复现 ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

## 2. Hyperparameters (改这里来调参)

In [ ]:
# ── 可以调整的超参数 ──
IMAGE_SIZE   = 224        # 预训练模型标准输入尺寸
BATCH_SIZE   = 32
NUM_EPOCHS   = 30
LR_HEAD      = 1e-3       # 新分类层的学习率
LR_BACKBONE  = 1e-4       # 预训练主干网络的学习率（小一点）
WEIGHT_DECAY = 1e-4
NUM_CLASSES  = 100
VAL_SPLIT    = 0.15       # 15% 数据用于验证

# ── 模型选择：'resnet50', 'efficientnet_b3', 'efficientnet_b0' ──
MODEL_NAME = 'efficientnet_b3'

## 3. Data Transforms & Dataset

In [ ]:
# ImageNet 均值和标准差（预训练模型的标准）
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ImageFolder 自动按文件夹名排序："0"->0, "1"->1, ... "99"->99
# 这正好满足作业要求的 label 顺序
full_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
print(f'Total training images: {len(full_dataset)}')
print(f'Classes (first 5): {full_dataset.classes[:5]}')
print(f'Class to index (first 5): {dict(list(full_dataset.class_to_idx.items())[:5])}')

# Train / Val split
val_size   = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)
# val 集不做 augmentation
val_dataset.dataset = datasets.ImageFolder(TRAIN_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {train_size} | Val: {val_size}')

## 4. Model Definition

In [ ]:
def build_model(model_name: str, num_classes: int):
    """加载预训练模型，替换最后的分类层"""
    if model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        head_params = model.fc.parameters()
        backbone_params = [p for n, p in model.named_parameters() if 'fc' not in n]

    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        head_params = model.classifier.parameters()
        backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]

    elif model_name == 'efficientnet_b3':
        model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        head_params = model.classifier.parameters()
        backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]

    else:
        raise ValueError(f'Unknown model: {model_name}')

    return model, head_params, backbone_params


model, head_params, backbone_params = build_model(MODEL_NAME, NUM_CLASSES)
model = model.to(DEVICE)
print(f'Model: {MODEL_NAME} loaded on {DEVICE}')

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## 5. Optimizer, Scheduler & Loss

In [ ]:
# 两种不同学习率：新分类层用大 lr，预训练主干用小 lr
optimizer = optim.AdamW([
    {'params': head_params,     'lr': LR_HEAD},
    {'params': backbone_params, 'lr': LR_BACKBONE},
], weight_decay=WEIGHT_DECAY)

# Cosine annealing：让学习率在训练过程中平滑下降
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print('Optimizer, scheduler, and loss function ready.')

## 6. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


# ── 训练主循环 ──
best_val_acc = 0.0
SAVE_PATH = '/content/best_model.pth'
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # 保存最好的模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)

    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}'
          + (' ← best' if val_acc == best_val_acc else ''))

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')

## 7. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].legend()

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print('Saved to /content/training_curves.png')

## 8. Generate Predictions & submission.csv

In [ ]:
# 加载最好的模型权重
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# 读 sample_submission 来拿到 ID 顺序
sample_df = pd.read_csv(SAMPLE_SUBMISSION)
print(f'Test images to predict: {len(sample_df)}')

predictions = []
with torch.no_grad():
    for img_id in sample_df['ID']:
        img_path = os.path.join(TEST_DIR, f'{img_id}.jpg')
        img = Image.open(img_path).convert('RGB')
        tensor = test_transform(img).unsqueeze(0).to(DEVICE)
        pred = model(tensor).argmax(1).item()
        predictions.append(pred)

sample_df['Label'] = predictions
SUBMISSION_PATH = '/content/submission.csv'
sample_df.to_csv(SUBMISSION_PATH, index=False)
print(f'submission.csv saved! Preview:')
print(sample_df.head())

## 9. Save Model Weights to Google Drive

In [ ]:
import shutil

DRIVE_SAVE_DIR = '/content/drive/MyDrive/cse144_outputs'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

shutil.copy(SAVE_PATH, os.path.join(DRIVE_SAVE_DIR, 'best_model.pth'))
shutil.copy(SUBMISSION_PATH, os.path.join(DRIVE_SAVE_DIR, 'submission.csv'))
shutil.copy('/content/training_curves.png', os.path.join(DRIVE_SAVE_DIR, 'training_curves.png'))

print(f'Files saved to Google Drive: {DRIVE_SAVE_DIR}')

## 10. (Optional) Test-Time Augmentation (TTA) — 提升准确率用

TTA：对每张测试图做多次不同的变换，取平均，可以提升 1-3% 准确率。

In [ ]:
tta_transforms = [
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((int(IMAGE_SIZE*1.1), int(IMAGE_SIZE*1.1))),
                        transforms.CenterCrop(IMAGE_SIZE),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
]

model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

tta_predictions = []
with torch.no_grad():
    for img_id in sample_df['ID']:
        img_path = os.path.join(TEST_DIR, f'{img_id}.jpg')
        img = Image.open(img_path).convert('RGB')

        # 对每种 transform 取 softmax 分数，取平均
        avg_probs = torch.zeros(NUM_CLASSES).to(DEVICE)
        for tfm in tta_transforms:
            tensor = tfm(img).unsqueeze(0).to(DEVICE)
            probs = torch.softmax(model(tensor), dim=1).squeeze(0)
            avg_probs += probs
        avg_probs /= len(tta_transforms)
        tta_predictions.append(avg_probs.argmax().item())

sample_df['Label'] = tta_predictions
TTA_PATH = '/content/submission_tta.csv'
sample_df.to_csv(TTA_PATH, index=False)
print('TTA submission saved:', TTA_PATH)
print(sample_df.head())